In [ ]:
import numpy as np
import cupy as cp
from scipy import ndimage
from types import SimpleNamespace
from scipy.fft import fftn, ifftn  # faster than np.fft on CPU

from holotomocupy.rec import Rec
from holotomocupy.shift import Shift
from holotomocupy.tomo_batched import TomoBatched
from holotomocupy.utils import *

# Acquisition parameters

In [ ]:
n = 128
ntheta = 360
detector_pixelsize = 1.4760147601476e-6 * 2 * 2048/n
energy = 17.1
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.217

sx0 = -3.135e-3
z1 = np.array([5.110, 5.464, 6.879, 9.817, 10.372, 11.146, 12.594, 17.209]) * 1e-3 - sx0
z1_ids = np.array([0, 1, 2, 3]) ### note using index starting from 0
z1 = z1[z1_ids]
ndist = len(z1)
z2 = focustodetectordistance - z1
distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]

nobj = int(np.ceil(n / norm_magnifications[-1] / 64)) * 64 
theta = cp.linspace(0,np.pi,ntheta).astype('float32')


# Data modeling parameters

In [ ]:
rec_args = SimpleNamespace(
    n = n,
    nz = n,
    nobj = nobj,
    nzobj = nobj,
    detector_pixelsize = detector_pixelsize,      
    ndist=ndist,
    ntheta=ntheta,
    energy=energy,        
    focustodetectordistance=focustodetectordistance,        
    z1=z1,   
    theta=theta,
    ngpus=1,
    nchunk = 4,
    rho=[1,0.05,0.02], ### should be sent for rec isntead
    mask=1.1,   
    obj_dtype='complex64',    
)


# Modeling data

In [ ]:
# read vars
obj = np.load(f'data/obj_{n}.npy')
prb = cp.load(f'data/prb_{n}.npy')
pos = np.load(f'data/pos_{n}.npy')


pos_err = (np.random.random(pos.shape).astype('float32')-0.5)*np.amax(pos)/16

# gen data
cl = Rec(rec_args)
ewave = cl.fwd_tomo(obj,exp=True)
data = np.abs(cl.fwd_ewave(ewave,pos+pos_err,prb))**2

# gen ref
ref = np.abs(cl.fwd_ewave(ewave*0+1,pos+pos_err,prb))**2
ref=ref[0]
print(pos[0,-1])
print(pos[100,-1])
mshow_polar(ewave[0],True)
mshow_complex(data[0,0]+1j*data[0,-1],True)
mshow_complex(ref[0]+1j*ref[-1],True)



In [ ]:
mshow_pos(pos,True)
mshow_pos(pos_err,True)

# Save data

In [ ]:
np.save(f'data/data_{n}',data)
np.save(f'data/ref_{n}',ref)
np.save(f'data/pos_err_{n}',pos_err)


